In [1]:
# 20-EPOCH IMPROVED VERSION (changed for faster convergence + better pruning)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

torch.manual_seed(42)

#########################################
# PRUNABLE CONV
#########################################

class PrunableConv(nn.Module):
    def __init__(self,in_c,out_c,k,pad=1):
        super().__init__()

        self.weight=nn.Parameter(
            torch.empty(out_c,in_c,k,k)
        )

        nn.init.kaiming_normal_(
            self.weight
        )

        self.bias=nn.Parameter(
            torch.zeros(out_c)
        )

        # changed: start open
        self.gate_scores=nn.Parameter(
            torch.ones_like(self.weight)*3.0
        )

        self.pad=pad

    def gates(self):
        return torch.sigmoid(
            self.gate_scores
        )

    def forward(self,x):

        if self.training:
            w=self.weight*self.gates()
        else:
            hard=(self.gates()>.5).float()
            w=self.weight*hard

        return F.conv2d(
            x,w,self.bias,padding=self.pad
        )


#########################################
# MODEL
#########################################

class SparseCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1=PrunableConv(3,64,3)
        self.bn1=nn.BatchNorm2d(64)

        self.conv2=PrunableConv(64,128,3)
        self.bn2=nn.BatchNorm2d(128)

        self.conv3=PrunableConv(128,256,3)
        self.bn3=nn.BatchNorm2d(256)

        self.fc1=nn.Linear(
            256*4*4,
            512
        )

        self.drop=nn.Dropout(.35)

        self.fc2=nn.Linear(
            512,
            10
        )


    def forward(self,x):

        x=F.relu(
            self.bn1(
                self.conv1(x)
            )
        )
        x=F.max_pool2d(x,2)

        x=F.relu(
            self.bn2(
                self.conv2(x)
            )
        )
        x=F.max_pool2d(x,2)

        x=F.relu(
            self.bn3(
                self.conv3(x)
            )
        )
        x=F.max_pool2d(x,2)

        x=x.view(
            x.size(0),-1
        )

        x=F.relu(
            self.fc1(x)
        )

        x=self.drop(x)

        return self.fc2(x)


    def sparsity_loss(self):

        penalties=[]

        for m in self.modules():
            if isinstance(
                m,
                PrunableConv
            ):
                penalties.append(
                    m.gates().mean()
                )

        return sum(
            penalties
        )/len(
            penalties
        )


    def sparsity_percent(self):

        total=0
        pruned=0

        for m in self.modules():

            if isinstance(
                m,
                PrunableConv
            ):

                g=m.gates()

                total+=g.numel()

                pruned+=(
                    g<.5
                ).sum().item()

        return 100*pruned/total


#########################################
# DATA
#########################################

train_transform=transforms.Compose([
transforms.RandomCrop(32,padding=4),
transforms.RandomHorizontalFlip(),
transforms.AutoAugment(),
transforms.ToTensor(),
transforms.Normalize(
(0.4914,0.4822,0.4465),
(0.247,0.243,0.261)
)
])

test_transform=transforms.Compose([
transforms.ToTensor(),
transforms.Normalize(
(0.4914,0.4822,0.4465),
(0.247,0.243,0.261)
)
])


trainset=torchvision.datasets.CIFAR10(
"./data",
train=True,
download=True,
transform=train_transform
)

testset=torchvision.datasets.CIFAR10(
"./data",
train=False,
download=True,
transform=test_transform
)

trainloader=DataLoader(
trainset,
batch_size=256,
shuffle=True,
num_workers=2
)

testloader=DataLoader(
testset,
batch_size=256,
shuffle=False,
num_workers=2
)


#########################################
# SETUP
#########################################

device="cuda" if torch.cuda.is_available() else "cpu"

model=SparseCNN().to(device)

EPOCHS=20
LAMBDA=.15


# changed:
# separate learning rates
weight_params=[]
gate_params=[]

for n,p in model.named_parameters():
    if "gate_scores" in n:
        gate_params.append(p)
    else:
        weight_params.append(p)


optimizer=optim.AdamW([
{"params":weight_params,"lr":0.002},
{"params":gate_params,"lr":0.01}
],
weight_decay=5e-4
)


scheduler=optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=.02,
steps_per_epoch=len(trainloader),
epochs=EPOCHS
)



#########################################
# EVAL
#########################################

def evaluate():

    model.eval()

    correct=0
    total=0

    with torch.no_grad():

        for x,y in testloader:

            x=x.to(device)
            y=y.to(device)

            pred=model(
                x
            ).argmax(1)

            correct+=(
                pred==y
            ).sum().item()

            total+=y.size(0)

    return 100*correct/total



#########################################
# TRAIN
#########################################

best=0

for epoch in range(EPOCHS):

    model.train()

    correct=0
    total=0

    for x,y in trainloader:

        x=x.to(device)
        y=y.to(device)

        optimizer.zero_grad()

        out=model(x)

        ce=F.cross_entropy(
            out,y,
            label_smoothing=.1
        )

        sparse=model.sparsity_loss()

        loss=ce + LAMBDA*sparse

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()
        scheduler.step()

        pred=out.argmax(1)

        correct+=(
            pred==y
        ).sum().item()

        total+=y.size(0)


    train_acc=100*correct/total

    test_acc=evaluate()

    sparsity=model.sparsity_percent()

    if test_acc>best:
        best=test_acc
        torch.save(
            model.state_dict(),
            "best_pruned_model.pth"
        )


    print(
f"Epoch {epoch+1}/20 | "
f"Train {train_acc:.2f}% | "
f"Test {test_acc:.2f}% | "
f"Sparsity {sparsity:.2f}%"
)


print(
"\nBest Accuracy:",
best
)

100%|██████████| 170M/170M [00:07<00:00, 21.5MB/s] 


Epoch 1/20 | Train 28.73% | Test 51.82% | Sparsity 0.00%
Epoch 2/20 | Train 38.78% | Test 53.98% | Sparsity 0.00%
Epoch 3/20 | Train 41.57% | Test 54.59% | Sparsity 0.01%
Epoch 4/20 | Train 47.25% | Test 62.24% | Sparsity 0.07%
Epoch 5/20 | Train 50.97% | Test 57.41% | Sparsity 0.51%
Epoch 6/20 | Train 53.05% | Test 64.67% | Sparsity 2.16%
Epoch 7/20 | Train 55.00% | Test 69.60% | Sparsity 5.30%
Epoch 8/20 | Train 57.37% | Test 70.16% | Sparsity 9.49%
Epoch 9/20 | Train 58.40% | Test 71.23% | Sparsity 13.95%
Epoch 10/20 | Train 60.50% | Test 72.58% | Sparsity 18.29%
Epoch 11/20 | Train 62.45% | Test 70.98% | Sparsity 22.00%
Epoch 12/20 | Train 64.00% | Test 75.75% | Sparsity 25.19%
Epoch 13/20 | Train 65.26% | Test 77.70% | Sparsity 27.80%
Epoch 14/20 | Train 66.56% | Test 77.02% | Sparsity 29.73%
Epoch 15/20 | Train 67.99% | Test 78.57% | Sparsity 31.18%
Epoch 16/20 | Train 69.00% | Test 78.74% | Sparsity 32.16%
Epoch 17/20 | Train 70.28% | Test 79.25% | Sparsity 32.78%
Epoch 18/20 | 